<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# 1. Method choice and why

### Method Selected: Random Forest Classifier (`max_depth=6`, `n_estimators=100`)

For this classification task, we selected a **Random Forest Classifier**.

### Why Random Forest fits this dataset & lane:
1. **Handles Non-Linear Thresholds:** Search engine signals (like `avg_position` and `ctr`) exhibit non-linear interactions. For example, a drop in position from #2 to #5 carries severe traffic loss compared to a drop from #32 to #35. Tree-based ensembles naturally partition non-linear rank spaces without requiring manual feature transformations.
2. **Robust to Scale Differences:** Our feature set combines raw counts (`impressions_90d`, `days_since_last_update`) with continuous decimal ratios (`ctr`, `avg_position`). Random Forests do not rely on distance metrics, making them immune to varying feature scales.
3. **Controls Overfitting on Noisy Data:** Setting a moderate depth (`max_depth=6`) prevents individual trees from memorizing client-specific search patterns, ensuring strong generalization across unseen client domains.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Load dataset directly from raw repository
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

# Define binary classification target: 1 if declining, 0 otherwise
df["target"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Select core features available prior to outcome evaluation
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "clicks_90d"
]

# Quick sanity check on data shapes and balance
print("Dataset Shape:", df.shape)
print("Target Class Balance:\n", df["target"].value_counts(normalize=True))
print("Selected Features:", feature_cols)

Dataset Shape: (30000, 45)
Target Class Balance:
 target
1    0.542067
0    0.457933
Name: proportion, dtype: float64
Selected Features: ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'clicks_90d']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# 2. Split design

### Grouped Split Design: `GroupShuffleSplit` by `client_id` (80/20)

We implemented a **Grouped Split by `client_id`** (25 train clients, 7 test clients) to evaluate model performance across completely held-out client domains.

### Why this split is honest:
* **Prevents Intra-Client Data Leakage:** Pages belonging to the same client share domain authority, technical SEO infrastructure, and industry-specific seasonal trends. Randomly splitting rows would leak client-level baseline signals into the test set, artificially inflating performance metrics.
* **Simulates Real-World Deployment:** FlyRank operates across multi-client portfolios. Evaluating the model on unseen clients directly tests its capability to predict content decay on new client onboardings without relying on domain-specific history.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Instantiate grouped splitter by client_id to prevent intra-client data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Print split proportions and client counts
print(f"Train Set: {len(train_df)} rows across {train_df['client_id'].nunique()} clients")
print(f"Test Set:  {len(test_df)} rows across {test_df['client_id'].nunique()} clients")
print(f"Train Target Rate: {train_df['target'].mean():.4f}")
print(f"Test Target Rate:  {test_df['target'].mean():.4f}")

# Confirm no client overlap across splits
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])
assert len(train_clients.intersection(test_clients)) == 0, "DATA LEAKAGE ERROR: Client overlap detected!"
print("Grouped split validation passed: Zero client overlap between Train and Test splits.")

Train Set: 23837 rows across 25 clients
Test Set:  6163 rows across 7 clients
Train Target Rate: 0.5501
Test Target Rate:  0.5110
Grouped split validation passed: Zero client overlap between Train and Test splits.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# 3. Train + compare vs my baseline

### Performance Comparison Table (Held-Out Test Set)

| Metric | Week-4 Heuristic Baseline | Week-5 ML Model (Random Forest) | Net Improvement |
| :--- | :---: | :---: | :---: |
| **Accuracy** | 50.11% | **59.14%** | +9.03% |
| **Precision** | 52.52% | **56.57%** | +4.05% |
| **Recall** | 24.48% | **86.31%** | **+61.83%** |
| **F1 Score** | 0.3340 | **0.6834** | **+0.3494** |
| **ROC-AUC** | 0.4902 | **0.6291** | **+0.1389** |

### Key Takeaway
The Week-4 heuristic baseline missed over **75% of true declining content** (Recall: 24.48%). The Random Forest model captures **86.31% of decaying pages** while increasing overall accuracy to 59.14% and boosting ROC-AUC from near-random (0.4902) to 0.6291.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Compute Week-4 Baseline Predictions on Test Split
# Week-4 Baseline Logic: Flag as 1 (declining/action needed) if stale >= 180 or poor CTR on Page 1
baseline_pred_risk = (
    (test_df["days_since_last_update"] >= 180) |
    ((test_df["avg_position"] <= 10.0) & (test_df["ctr"] < 0.02))
).astype(int)

# Scaled Continuous Baseline Score for ROC-AUC
staleness_norm = (test_df["days_since_last_update"] - test_df["days_since_last_update"].min()) / (test_df["days_since_last_update"].max() - test_df["days_since_last_update"].min())
volume_norm = np.log1p(test_df["impressions_90d"]) / np.log1p(test_df["impressions_90d"].max())
baseline_score = ((staleness_norm * 0.5) + (volume_norm * 0.5))

# 2. Train ML Model (Random Forest Classifier)
X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test = test_df["target"]

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

model_pred_class = rf_model.predict(X_test)
model_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# 3. Compute Metrics Comparison Table
metrics_data = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Week-4 Baseline": [
        accuracy_score(y_test, baseline_pred_risk),
        precision_score(y_test, baseline_pred_risk, zero_division=0),
        recall_score(y_test, baseline_pred_risk, zero_division=0),
        f1_score(y_test, baseline_pred_risk, zero_division=0),
        roc_auc_score(y_test, baseline_score)
    ],
    "Week-5 ML Model (Random Forest)": [
        accuracy_score(y_test, model_pred_class),
        precision_score(y_test, model_pred_class, zero_division=0),
        recall_score(y_test, model_pred_class, zero_division=0),
        f1_score(y_test, model_pred_class, zero_division=0),
        roc_auc_score(y_test, model_pred_prob)
    ]
}

comp_df = pd.DataFrame(metrics_data)
print("=== MODEL VS BASELINE COMPARISON TABLE ===")
print(comp_df.to_string(index=False))

=== MODEL VS BASELINE COMPARISON TABLE ===
   Metric  Week-4 Baseline  Week-5 ML Model (Random Forest)
 Accuracy         0.501055                         0.591433
Precision         0.525204                         0.565661
   Recall         0.244840                         0.863131
 F1 Score         0.333983                         0.683430
  ROC-AUC         0.490185                         0.629118


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

# 4. Errors and interpretation

### Feature Importance Hierarchy
1. **`impressions_90d` (41.9%):** Total traffic exposure is the primary driver of decay vulnerability; higher impression pages face steeper organic risk.
2. **`avg_position` (31.8%):** SERP rank tier dictates the baseline click volume available for decay.
3. **`days_since_last_update` (13.7%):** Content age acts as a steady background risk factor.
4. **`clicks_90d` (6.4%) & `ctr` (6.1%):** Fine-grained engagement conversions refine risk predictions.

### Error Analysis & Model Weaknesses

* **False Positives ($n = 2,087$):** The model predicts decay for stable/growing pages when they exhibit low update frequencies ($\mu = 34.6$ days) combined with lower SERP ranks ($\mu = 15.8$). These pages appear structurally vulnerable to the model despite holding stable keyword niches.
* **False Negatives ($n = 431$):** The model fails to flag decay on ultra-high traffic pages ($\mu = 9,135$ impressions) that sit further back in rankings ($\mu = 20.8$). Because their impression volume is high, the model misclassifies them as healthy, missing stealth decay occurring on low-ranking high-volume terms.

### Operational Conclusion
The model heavily prioritizes **Recall over Precision**, making it an effective top-of-funnel filter for editorial queues. False positives can easily be triaged by editors, while false negatives are reduced by 61% compared to the Week-4 rule.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Feature Importance Breakdown
importances = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("=== FEATURE IMPORTANCES ===")
print(importances.to_string(index=False))

# 2. Error Analysis (False Positives vs. False Negatives)
test_eval = test_df.copy()
test_eval["pred_class"] = model_pred_class
test_eval["pred_prob"] = model_pred_prob

fp = test_eval[(test_eval["target"] == 0) & (test_eval["pred_class"] == 1)]
fn = test_eval[(test_eval["target"] == 1) & (test_eval["pred_class"] == 0)]

print("\n=== ERROR ANALYSIS SUMMARY ===")
print(f"Total Test Samples: {len(test_eval)}")
print(f"False Positives (Predicted Decay, Actually Stable/Growing): {len(fp)}")
print(f"False Negatives (Predicted Stable, Actually Decaying): {len(fn)}")

print("\nAverage Stats for False Positives (FP):")
print(fp[["days_since_last_update", "impressions_90d", "avg_position", "ctr"]].mean().to_string())

print("\nAverage Stats for False Negatives (FN):")
print(fn[["days_since_last_update", "impressions_90d", "avg_position", "ctr"]].mean().to_string())

=== FEATURE IMPORTANCES ===
               Feature  Importance
       impressions_90d    0.419058
          avg_position    0.318460
days_since_last_update    0.137227
            clicks_90d    0.064173
                   ctr    0.061082

=== ERROR ANALYSIS SUMMARY ===
Total Test Samples: 6163
False Positives (Predicted Decay, Actually Stable/Growing): 2087
False Negatives (Predicted Stable, Actually Decaying): 431

Average Stats for False Positives (FP):
days_since_last_update      34.640153
impressions_90d           3910.479636
avg_position                15.814039
ctr                          0.252300

Average Stats for False Negatives (FN):
days_since_last_update      40.909513
impressions_90d           9135.290023
avg_position                20.759861
ctr                          0.476218


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.